# 04 — Question 3 feasibility: are there enough occupation changers?

Question 3 relies on **within-person** comparisons: the link between exposure and wages is identified only by individuals whose occupation, and therefore exposure, changes over time.
Before estimating anything, we check that there are enough of them.

**Rule:** only counts and summary statistics are displayed.

**Threshold:** at least 200 to 300 individuals changing occupation at the 2-digit ISCO level are needed for fixed-effects estimates to be reasonably precise.

In [ ]:
import pandas as pd

df = pd.read_parquet("../Data/private/shp_isco4_ilo.parquet")
df = df[df["exposure_source"] != "none"]
df["isco2"] = df["isco4"].str[:2]
df["isco1"] = df["isco4"].str[:1]

tert = df[df["tertiary"] == 1]

## 1. Occupation changers at three levels of detail
- **`isco4`**: upper bound, includes coding noise (the same job coded `2512` one year and `2514` the next);
- **`isco2`**: more credible changes (a different type of occupation);
- **`isco1`**: changes of major group, almost certainly real, but rarer.

In [ ]:
n_years = tert.groupby("idpers")["year"].nunique()
print("Tertiary graduates:", tert["idpers"].nunique())
print("... observed at least 2 years:", (n_years >= 2).sum())

for level in ["isco4", "isco2", "isco1"]:
    n_occ = tert.groupby("idpers")[level].nunique()
    print(f"{level}: {(n_occ > 1).sum()} individuals with more than one code ({(n_occ > 1).mean():.1%})")

## 2. Removing vague codes and measuring the size of exposure changes
Vague codes padded with zeros (`exposure_source == "group"`, e.g. `2000` → `2512`) create spurious changes even at the 2-digit level, so they are removed.
We then measure how much exposure actually changes from one observed year to the next for the same individual: if changes were tiny (median below 0.02 on a 0–1 scale), any wage effect would be undetectable.

In [ ]:
precise = tert[tert["exposure_source"] == "isco4"]
n_occ = precise.groupby("idpers")["isco2"].nunique()
print("2-digit ISCO changers, precise codes only:", (n_occ > 1).sum())

precise = precise.sort_values(["idpers", "year"])
delta = precise.groupby("idpers")["exposure_main"].diff().abs()
changes = delta[delta > 0]
print("Number of exposure changes:", len(changes))
print(changes.describe().round(3))

## 3. All workers (robustness sample)

In [ ]:
n_occ_all = df.groupby("idpers")["isco2"].nunique()
print("2-digit ISCO changers, all workers:", (n_occ_all > 1).sum())

## 4. Conclusion

- Among 5,360 tertiary graduates (4,188 observed at least two years), 835 change 2-digit ISCO code; 623 remain once vague codes are removed, well above the threshold of 200 to 300.
- Exposure changes are substantial: median 0.09 on a 0–1 scale (mean 0.11, 75th percentile 0.15), so there is enough variation to detect an association with wages.
- Removing vague codes eliminates about 200 spurious changers (835 → 623), which justifies restricting Q3 to precise codes.
- Including all workers raises the number of 2-digit changers to 1,507, which is used as a robustness sample.

**Decision:** Question 3 is feasible. Main specification on tertiary graduates with precise codes; all workers as robustness check.